In [30]:
import numpy as np
import torch
import torch.nn as nn
import scipy.stats

# LSM

In [45]:
class LSMOptionPricer:
    def __init__(self, d=1, S0=100.0, K=100.0, r=0.05, delta=0.0,sigma=0.20, T=3.0, n_steps=10, n_paths=100000):
        self.d = d
        self.S0 = S0 if isinstance(S0, np.ndarray) else np.ones(d) * S0
        self.K = K
        self.r = r # riskless rate
        self.delta = delta
        self.sigma = sigma
        self.T = T
        self.n_steps = n_steps
        self.n_paths = n_paths
        self.dt = T / n_steps
        self.rng = np.random.default_rng(42)

    def simulate_paths(self):
        paths = np.zeros((self.n_steps + 1, self.d, self.n_paths))
        paths[0, :, :] = self.S0[:, np.newaxis]
        
        for i in range(1, self.n_steps + 1):
            Z = self.rng.standard_normal((self.d, self.n_paths))
            paths[i, :, :] = paths[i-1, :, :] * np.exp(
                (self.r - self.delta-0.5 * self.sigma**2) * self.dt + self.sigma * np.sqrt(self.dt) * Z
            )
        return paths

    def get_payoff(self, spots):
        
        max_spots = np.max(spots, axis=0)
        return np.maximum(max_spots - self.K, 0.0)

    def extract_features(self, current_spots, current_payoffs):
        if self.d == 1:
            return np.column_stack((
                np.ones(len(current_payoffs)),
                current_spots[0, :],
                current_spots[0, :]**2
            ))
        else:
            max_val = np.max(current_spots, axis=0)
            second_max = np.partition(current_spots, -2, axis=0)[-2, :] if self.d > 1 else max_val
            pay_val = current_payoffs
            
            return np.column_stack((
                np.ones(len(current_payoffs)),
                max_val, max_val**2,
                second_max,
                pay_val
            ))

    def price(self):
        paths = self.simulate_paths()
        cash_flows = self.get_payoff(paths[-1, :, :])
        discount = np.exp(-self.r * self.dt)
        
        for i in range(self.n_steps - 1, 0, -1):
            current_spots = paths[i, :, :]
            current_payoffs = self.get_payoff(current_spots)
            
            itm_idx = current_payoffs > 0
            
            if np.sum(itm_idx) > 10:
                X_reg = self.extract_features(current_spots[:, itm_idx], current_payoffs[itm_idx])
                Y_reg = cash_flows[itm_idx] * discount
                
                coeffs, _, _, _ = np.linalg.lstsq(X_reg, Y_reg, rcond=None)
                continuation_value = X_reg @ coeffs
                exercise = current_payoffs[itm_idx] > continuation_value
                
                full_exercise_bool = np.zeros(self.n_paths, dtype=bool)
                full_exercise_bool[itm_idx] = exercise
                
                cash_flows[full_exercise_bool] = current_payoffs[full_exercise_bool]
                cash_flows[~full_exercise_bool] = cash_flows[~full_exercise_bool] * discount
            else:
                cash_flows = cash_flows * discount
        discounted_cash_flows = cash_flows * discount
        
        bermudan_price = np.mean(discounted_cash_flows)
        variance = np.var(discounted_cash_flows, ddof=1)
        standard_error = np.sqrt(variance) / np.sqrt(self.n_paths)

        bermudan =  {
            'price': bermudan_price,
            'variance': variance,
            'standard_error': standard_error
        }       
        
        
        terminal_payoff = self.get_payoff(paths[-1, :, :])
        european_price = np.exp(-self.r * self.T) * np.mean(terminal_payoff)
        
        return bermudan, european_price

def get_CI(bermudan,CI = 0.95):
    z = scipy.stats.norm.ppf(0.5+CI/2) 
    IC_inf = round(bermudan['price'] - z * bermudan['standard_error'], 3)
    IC_sup = round(bermudan['price'] + z * bermudan['standard_error'], 3)
    print((IC_inf,IC_sup))
if __name__ == "__main__":
    
    dimensions = [1,5,10,100]
    for dim in dimensions:
        pricer = LSMOptionPricer(d=dim, S0=100.0, K=100.0, r=0.05, delta=0.10, sigma=0.20, T=3.0, n_steps=10)
        bermudan, european = pricer.price()
        print(f" {dim}D Call - Bermudan: {bermudan['price']:.4f}, European: {european:.4f}, Premium: {bermudan['price']- european:.4f}")
        get_CI(bermudan)
    

 1D Call - Bermudan: 7.9802, European: 6.1021, Premium: 1.8782
(7.906, 8.054)
 5D Call - Bermudan: 26.1271, European: 23.0546, Premium: 3.0725
(26.001, 26.253)
 10D Call - Bermudan: 38.2320, European: 35.5981, Premium: 2.6339
(38.094, 38.37)
 100D Call - Bermudan: 83.7014, European: 82.1239, Premium: 1.5775
(83.553, 83.85)


# Deep Optimal Stopping

In [38]:
import scipy.stats
np.random.seed(42)

class stock:
    def __init__(self, T, K, sigma, delta, So, r, N, M, d):
        self.T = T
        self.K = K
        self.sigma = sigma * np.ones(d)
        self.delta = delta
        self.So = So*np.ones(d)
        self.r = r
        self.N = N
        self.M = M
        self.d = d

    def GBM(self):

        dt = self.T/self.N
        So_vec = self.So*np.ones((1, self.M, self.d))

        Z = np.random.standard_normal((self.N, self.M, self.d))
        s = self.So*np.exp(np.cumsum((self.r-self.delta-0.5 *
                           self.sigma**2)*dt+self.sigma*np.sqrt(dt)*Z, axis=0))
        s = np.append(So_vec, s, axis=0)
        return s

    def g(self, n, m, X):
        max1 = torch.max(X[int(n), m, :].float()-self.K)
        return np.exp(-self.r * (self.T/self.N)*n)*torch.max(max1, torch.tensor([0.0]))


# %%
class NeuralNet(torch.nn.Module):
    def __init__(self, d, q1, q2):
        super(NeuralNet, self).__init__()
        self.a1 = nn.Linear(d, q1)
        self.relu = nn.ReLU()
        self.a2 = nn.Linear(q1, q2)
        self.a3 = nn.Linear(q2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.a1(x)
        out = self.relu(out)
        out = self.a2(out)
        out = self.relu(out)
        out = self.a3(out)
        out = self.sigmoid(out)

        return out


def loss(y_pred, s, x, n, tau):
    r_n = torch.zeros((s.M))
    for m in range(0, s.M):
        r_n[m] = -s.g(n, m, x)*y_pred[m] - s.g(tau[m], m, x)*(1-y_pred[m])
    return (r_n.mean())

S = stock(T=3.0, K=100, sigma=0.20, delta=0.1, So=100.0, r=0.05, N=9, M=8192, d=2)

X = torch.from_numpy(S.GBM()).float()
# %%

def NN(n, x, s, tau_n_plus_1):
    epochs = 30
    model = NeuralNet(s.d, s.d+40, s.d+40)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    for _ in range(epochs):
        F = model.forward(X[n])
        optimizer.zero_grad()
        criterion = loss(F, S, X, n, tau_n_plus_1)
        criterion.backward()
        optimizer.step()

    return F, model


mods = [None]*S.N
tau_mat = np.zeros((S.N+1, S.M))
tau_mat[S.N, :] = S.N

f_mat = np.zeros((S.N+1, S.M))
f_mat[S.N, :] = 1

# %%
for n in range(S.N-1, -1, -1):
    probs, mod_temp = NN(n, X, S, torch.from_numpy(tau_mat[n+1]).float())
    mods[n] = mod_temp
    np_probs = probs.detach().numpy().reshape(S.M)
    print(n, ":", np.min(np_probs), " , ", np.max(np_probs))

    f_mat[n, :] = (np_probs > 0.5) * 1.0
    tau_mat[n, :] = np.argmax(f_mat, axis=0)

# %%
Y = torch.from_numpy(S.GBM()).float()

tau_mat_test = np.zeros((S.N+1, S.M))
tau_mat_test[S.N, :] = S.N

f_mat_test = np.zeros((S.N+1, S.M))
f_mat_test[S.N, :] = 1

V_mat_test = np.zeros((S.N+1, S.M))
V_est_test = np.zeros(S.N+1)

for m in range(0, S.M):
    V_mat_test[S.N, m] = S.g(S.N, m, Y)

V_est_test[S.N] = np.mean(V_mat_test[S.N, :])

for n in range(S.N-1, -1, -1):
    mod_curr = mods[n]
    probs = mod_curr(Y[n])
    np_probs = probs.detach().numpy().reshape(S.M)

    f_mat_test[n, :] = (np_probs > 0.5)*1.0

    tau_mat_test[n, :] = np.argmax(f_mat_test, axis=0)

    for m in range(0, S.M):
        V_mat_test[n, m] = np.exp(
            (n-tau_mat_test[n, m])*(-S.r*S.T/S.N))*S.g(tau_mat_test[n, m], m, X)


# %%
V_est_test = np.mean(V_mat_test, axis=1)
V_std_test = np.std(V_mat_test, axis=1)
V_se_test = V_std_test/(np.sqrt(S.M))

z = scipy.stats.norm.ppf(0.975)
lower = V_est_test[0] - z*V_se_test[0]
upper = V_est_test[0] + z*V_se_test[0]

print(V_est_test[0])
print(V_se_test[0])
print(lower)
print(upper)

8 : 9.498734e-06  ,  0.9986773
7 : 9.682579e-23  ,  0.001128238
6 : 0.5795509  ,  0.99994683
5 : 8.5409874e-11  ,  0.010157617
4 : 0.43890277  ,  0.9936272
3 : 0.0005631853  ,  0.89549893
2 : 0.9229859  ,  0.999974
1 : 1.5369079e-06  ,  0.058909453
0 : 1.6074594e-08  ,  1.6074594e-08
8.755050610568105
0.12324562046418272
8.513493633206014
8.996607587930196
